## RESET OUTPUT MODEL SAVE LOCATION BEFORE RUNNING 


In [1]:
from pathlib import Path
import re, itertools

# --- set these to your v2 paths ---
TRAIN_DIR   = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/train_hires")
TRAIN_T1    = TRAIN_DIR / "t1"
TRAIN_MASKS = TRAIN_DIR / "masks"

assert TRAIN_T1.exists() and TRAIN_MASKS.exists(), "t1/ or masks/ folder is missing."

# must mirror the loader’s suffix logic
IMAGE_SUFFIXES = ("_T1w_MNI_norm", "_T1w_MNI", "_T1w_brain", "_T1w", "_T1", "_img_prepped")
MASK_SUFFIXES  = ("_lesion_mask_MNI_clean", "_lesion_mask_MNI", "_lesion_mask", "_desc-lesion_mask", "_mask", "_mask_prepped")
CLEANUP = ("_img_prepped", "_mask_prepped", "_image", "_img")

def strip_ext(name):  # .nii.gz-aware
    return name[:-7] if name.endswith(".nii.gz") else Path(name).stem

def strip_suffix(stem, suffixes):
    for s in sorted(suffixes, key=len, reverse=True):
        if stem.endswith(s):
            return stem[:-len(s)]
    return stem

def normalize_key(name, suffixes):
    stem = strip_ext(name)
    for c in CLEANUP:
        if stem.endswith(c):
            stem = stem[:-len(c)]
    stem = strip_suffix(stem, suffixes)
    return stem.rstrip("_")

imgs  = sorted([p for p in TRAIN_T1.glob("*.nii.gz")])
msks  = sorted([p for p in TRAIN_MASKS.glob("*.nii.gz")])

img_map = {normalize_key(p.name, IMAGE_SUFFIXES): p for p in imgs}
msk_map = {normalize_key(p.name, MASK_SUFFIXES): p for p in msks}

ik = list(itertools.islice(img_map.items(), 10))
mk = list(itertools.islice(msk_map.items(), 10))

print(f"Images: {len(imgs)}   Masks: {len(msks)}")
print("Sample normalized IMG keys -> file:")
for k,v in ik: print("  ", k, "->", v.name)
print("Sample normalized MSK keys -> file:")
for k,v in mk: print("  ", k, "->", v.name)

inter = sorted(set(img_map).intersection(msk_map))
print(f"\nKey intersection count: {len(inter)}")
print("First 10 intersecting keys:", inter[:10])


Images: 323   Masks: 323
Sample normalized IMG keys -> file:
   sub-M2012_ses-1158 -> sub-M2012_ses-1158_T1w_MNI_norm.nii.gz
   sub-M2018_ses-341 -> sub-M2018_ses-341_T1w_MNI_norm.nii.gz
   sub-M2026_ses-3098 -> sub-M2026_ses-3098_T1w_MNI_norm.nii.gz
   sub-M2034_ses-1568 -> sub-M2034_ses-1568_T1w_MNI_norm.nii.gz
   sub-M2035_ses-4295 -> sub-M2035_ses-4295_T1w_MNI_norm.nii.gz
   sub-M2036_ses-695 -> sub-M2036_ses-695_T1w_MNI_norm.nii.gz
   sub-M2042_ses-1953 -> sub-M2042_ses-1953_T1w_MNI_norm.nii.gz
   sub-M2043_ses-183 -> sub-M2043_ses-183_T1w_MNI_norm.nii.gz
   sub-M2045_ses-381 -> sub-M2045_ses-381_T1w_MNI_norm.nii.gz
   sub-M2047_ses-553 -> sub-M2047_ses-553_T1w_MNI_norm.nii.gz
Sample normalized MSK keys -> file:
   sub-M2012_ses-1158 -> sub-M2012_ses-1158_lesion_mask_MNI_clean.nii.gz
   sub-M2018_ses-341 -> sub-M2018_ses-341_lesion_mask_MNI_clean.nii.gz
   sub-M2026_ses-3098 -> sub-M2026_ses-3098_lesion_mask_MNI_clean.nii.gz
   sub-M2034_ses-1568 -> sub-M2034_ses-1568_lesion_mask_

In [ ]:
# === Fresh run: small-lesion–biased, gentler aug, longer schedule ===
from pathlib import Path
import importlib.util, os, sys, gc, time, traceback, shlex, subprocess
import tensorflow as tf
from tensorflow.keras import mixed_precision

# ----------- Paths -----------
CUDA_ID        = "0"
ROOT_V2        = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v2")
MODULE_PATH    = Path("/home/rbielski/stroke_cleaned/stroke_segmentation_v1.2/stroke_seg_v1.2_train.py")
TRAIN_DIR      = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/train_hires")
TRAIN_T1       = TRAIN_DIR / "t1"
TRAIN_MASKS    = TRAIN_DIR / "masks"

RUN_ID         = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR        = ROOT_V2 / "runs" / RUN_ID
MODEL_DIR      = RUN_DIR / "models"
CALLBACKS_DIR  = RUN_DIR / "callbacks"
LOG_DIR        = RUN_DIR / "logs"
for d in (MODEL_DIR, CALLBACKS_DIR, LOG_DIR): d.mkdir(parents=True, exist_ok=True)

# ----------- Env / session -----------
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_ID
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["SMARTSOTA_LOG_DIR"] = str(LOG_DIR)

tf.keras.backend.clear_session(); gc.collect()
mixed_precision.set_global_policy("mixed_float16")

# Tee logs to notebook + file
class Tee:
    def __init__(self,*streams): self.streams=streams
    def write(self,d): 
        for s in self.streams: s.write(d); s.flush()
        return len(d)
    def flush(self): 
        for s in self.streams: s.flush()
log_file = open(LOG_DIR/"train_stdout_stderr.log","a", buffering=1)
sys.stdout = Tee(sys.__stdout__, log_file)
sys.stderr = Tee(sys.__stderr__, log_file)

print("Run ID:", RUN_ID)
print("TF:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
for g in tf.config.list_physical_devices("GPU"):
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception as e: print("set_memory_growth failed:", e)

# ----------- Import training module (no MirroredStrategy) -----------
spec = importlib.util.spec_from_file_location("arc_seg_train", MODULE_PATH)
seg = importlib.util.module_from_spec(spec)
seg.tf = tf
spec.loader.exec_module(seg)
seg.strategy = tf.distribute.get_strategy()
print("Strategy:", type(seg.strategy).__name__)

# ----------- Hyperparameters (from scratch) -----------
INPUT_SHAPE   = (192, 224, 192, 1)
BATCH_SIZE    = 1
BASE_FILTERS  = 8
SAM_HEADS     = 2
VAL_SPLIT     = 0.15

# Loss emphasis for small/thin structures
DICE_WEIGHT       = 0.30
BOUNDARY_WEIGHT   = 0.70

# Gentler but nonzero augmentation
AUG_INTENSITY = 0.15
ROTATION_RANGE = 15

# Longer cosine schedule with warmup
TOTAL_EPOCHS  = 120
INITIAL_EPOCH = 0
INITIAL_LR    = 1e-4
MIN_LR        = 3e-6
WARMUP_EPOCHS = 10

# If your generator honors this, it biases toward small lesions
SMALL_LESION_THRESHOLD = 25000

# ----------- Launch training (fresh; no resume) -----------
try:
    history = seg.train_dynamic_model(
        # data roots; use separated subfolders to avoid duplicates
        DATA_DIR=TRAIN_DIR,
        IMAGES_DIR=TRAIN_T1,
        MASKS_DIR=TRAIN_MASKS,

        MODEL_DIR=MODEL_DIR,
        CALLBACKS_DIR=CALLBACKS_DIR,

        TOTAL_EPOCHS=TOTAL_EPOCHS,
        INITIAL_EPOCH=INITIAL_EPOCH,
        LOAD_WEIGHTS_FROM=None,
        RESUME_FROM_LATEST=False,

        INPUT_SHAPE=INPUT_SHAPE,
        BATCH_SIZE=BATCH_SIZE,
        BASE_FILTERS=BASE_FILTERS,
        SAM_HEADS=SAM_HEADS,
        RESAMPLE_TO_TARGET=True,

        VALIDATION_SPLIT=VAL_SPLIT,
        AUGMENTATION_INTENSITY=AUG_INTENSITY,
        ROTATION_RANGE=ROTATION_RANGE,

        # loss balance
        DICE_WEIGHT=DICE_WEIGHT,
        BOUNDARY_WEIGHT=BOUNDARY_WEIGHT,

        # schedule
        INITIAL_LR=INITIAL_LR,
        MIN_LR=MIN_LR,
        WARMUP_EPOCHS=WARMUP_EPOCHS,

        # small-lesion bias (if implemented in your gen)
        SMALL_LESION_THRESHOLD=SMALL_LESION_THRESHOLD,
    )
    print("Training complete. Logged keys:", list(getattr(history, "history", {}).keys()))
    print("Run dir:", RUN_DIR)

except Exception:
    print("\n================= UNCAUGHT EXCEPTION =================")
    traceback.print_exc()
    print("======================================================\n")
    try:
        print("Last few GPU snapshots:")
        for _ in range(3):
            subprocess.run(shlex.split("nvidia-smi"), check=False)
            time.sleep(1)
    except Exception:
        pass
    raise
finally:
    try: log_file.flush()
    except: pass


2025-11-07 10:33:54.837711: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Run ID: 20251107_103356
TF: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Strategy: _DefaultDistributionStrategy
Strategy: _DefaultDistributionStrategy


2025-11-07 10:33:56,684 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2025-11-07 10:33:56,684 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2025-11-07 10:33:56,684 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlow 2.20.0
- NumPy 2.2.1
- GPU devices: 1
2025-11-07 10:33:56,687 - SmartSOTA_Dynamic - INFO - 🧭 INPUT_SHAPE set to: (192, 224, 192, 1)
2025-11-07 10:33:56,687 - SmartSOTA_Dynamic - INFO - 📚 Loading dataset (flex loader for T1w volumes)…
I0000 00:00:1762536836.788394 1302733 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1762536836.789416 1302733 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22148 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
2025-11-07 10:33:56,792 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_start: CPU=0.73GB | GPU mem track

2025-11-07 10:34:39,626 - SmartSOTA_Dynamic - INFO - Model: "SmartSOTA_Dynamic"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 192, 224,  │          0 │ -                 │
│ (InputLayer)        │ 192, 1)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_conv_block │ (None, 192, 224,  │      2,024 │ input_layer[0][0] │
│ (ResidualConvBlock) │ 192, 8)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vision_mamba_block  │ (None, 192, 224,  │      7,192 │ residual_conv_bl… │
│ (VisionMambaBlock)  │ 192, 8)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

Epoch 1/120


...

2025-11-07 12:53:07,682 - SmartSOTA_Dynamic - INFO - Memory at epoch_119_end: CPU=11.88GB | GPU mem tracking failed | Disk: 1231.1GB free


Epoch 120: dice=0.4300 val_dice=0.5406 loss=0.1758 val_loss=0.1429 lr=3.00e-06
258/258 ━━━━━━━━━━━━━━━━━━━━ 69s 266ms/step - dice_coefficient: 0.4300 - loss: 0.1758 - val_dice_coefficient: 0.5406 - val_loss: 0.1429 - learning_rate: 3.0000e-06


2025-11-07 12:53:08,217 - SmartSOTA_Dynamic - INFO - 💾 Saved final weights to /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v2/runs/20251107_103356/models/smart_sota_dynamic_20251107_103356.final.weights.h5


2025-11-07 12:53:08,768 - SmartSOTA_Dynamic - INFO - 💾 Saved full model to /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v2/runs/20251107_103356/models/smart_sota_dynamic_20251107_103356.keras
2025-11-07 12:53:08,768 - SmartSOTA_Dynamic - INFO - 🏁 Training complete.


Training complete. Logged keys: ['dice_coefficient', 'loss', 'val_dice_coefficient', 'val_loss', 'learning_rate']
Run dir: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v2/runs/20251107_103356


In [1]:
from pathlib import Path
import os, csv, json, re

RUN_ROOT = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v2")  # <-- v2 root

def latest_run_dir(run_root: Path) -> Path:
    runs_dir = run_root / "runs"
    candidates = [p for p in runs_dir.glob("*") if p.is_dir()]
    if not candidates:
        raise FileNotFoundError(f"No run folders found under {runs_dir}")
    return max(candidates, key=lambda p: p.stat().st_mtime)

def read_from_history_csv(cb_dir: Path):
    csv_path = cb_dir / "history.csv"
    if not csv_path.exists(): return None
    best_val = best_epoch = last_val = last_epoch = None
    with open(csv_path, newline="") as f:
        for i, row in enumerate(csv.DictReader(f)):
            v = row.get("val_dice_coefficient")
            if not v: continue
            val = float(v); last_val, last_epoch = val, i
            if best_val is None or val > best_val:
                best_val, best_epoch = val, i
    if last_val is None: return None
    return {"source":"CSV","last_val":last_val,"last_epoch":last_epoch,
            "best_val":best_val,"best_epoch":best_epoch,"path":str(csv_path)}

def read_from_history_json(cb_dir: Path):
    jpath = cb_dir / "artifacts" / "history_epoch.json"
    if not jpath.exists(): return None
    with open(jpath) as f: h = json.load(f)
    vals = h.get("val_dice_coefficient") or []
    if not vals: return None
    best_val = max(vals); best_epoch = vals.index(best_val)
    last_val = vals[-1]; last_epoch = len(vals) - 1
    return {"source":"JSON","last_val":last_val,"last_epoch":last_epoch,
            "best_val":best_val,"best_epoch":best_epoch,"path":str(jpath)}

def read_from_log(log_dir: Path):
    log_path = log_dir / "train_stdout_stderr.log"
    if not log_path.exists(): return None
    pat = re.compile(r"val_dice_coefficient:\s*([0-9]*\.?[0-9]+)")
    vals = []
    with open(log_path, "r", errors="ignore") as f:
        for line in f:
            m = pat.search(line)
            if m: vals.append(float(m.group(1)))
    if not vals: return None
    best_val = max(vals); best_epoch = vals.index(best_val)
    last_val = vals[-1]; last_epoch = len(vals) - 1
    return {"source":"LOG","last_val":last_val,"last_epoch":last_epoch,
            "best_val":best_val,"best_epoch":best_epoch,"path":str(log_path)}

run_dir = latest_run_dir(RUN_ROOT)
cb_dir  = run_dir / "callbacks"
log_dir = run_dir / "logs"

res = (read_from_history_csv(cb_dir)
       or read_from_history_json(cb_dir)
       or read_from_log(log_dir))

print(f"Using RUN_DIR: {run_dir}")
if res:
    print(f"[{res['source']}] last val_dice: {res['last_val']:.6f} (epoch {res['last_epoch']})")
    print(f"[{res['source']}] best  val_dice: {res['best_val']:.6f} (epoch {res['best_epoch']})")
    print(f"Source file: {res['path']}")
else:
    print("Could not find val_dice in CSV, JSON, or logs for this run.")


Using RUN_DIR: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v2/runs/20251107_103356
[CSV] last val_dice: 0.540626 (epoch 119)
[CSV] best  val_dice: 0.557842 (epoch 118)
Source file: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v2/runs/20251107_103356/callbacks/history.csv
